# 🎬 mosdif (MosaicDiff) - Video Mosaic Restoration
### Powered by BasicVSR++ & LTX-2.5 Uncensored Turbo (GGUF)

Notebook ini menjalankan pipeline restorasi video sensor/mozaik secara interaktif langsung di dalam Google Colab.
Semua input parameter (`#@param`), eksekusi, serta preview video sebelum dan sesudah (Base64) terintegrasi langsung di dalam satu notebook ini tanpa memerlukan Gradio atau tunnel eksternal.

> ⚡ **PENTING:** Pastikan runtime menggunakan GPU: **Runtime > Change runtime type > T4 / L4 / A100**.

### 1. Cek GPU dan Alokasi VRAM
Pastikan GPU NVIDIA terdeteksi dan memiliki alokasi memori yang cukup.

In [ ]:
!nvidia-smi
import torch
print('\n' + '='*50)
print('CUDA Available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name    :', torch.cuda.get_device_name(0))
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'Total VRAM     : {total_vram:.2f} GB')
print('='*50)

### 2. Setup Workspace & ComfyUI-GGUF Backend
Mengkloning repositori mosdif, ComfyUI backend, dan ComfyUI-GGUF custom node.

In [ ]:
# 1. Clone mosdif repository jika belum ada
import os
from pathlib import Path

if not os.path.exists('/content/mosdif'):
    !git clone https://github.com/azifypage/mosdif.git /content/mosdif
else:
    !git -C /content/mosdif pull || true

%cd /content/mosdif

# 2. Clone ComfyUI backend dan install dependensinya
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
    !pip install -q -r /content/ComfyUI/requirements.txt

# 3. Clone ComfyUI-GGUF custom node (Wajib untuk model LTX-2.5 GGUF)
if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-GGUF'):
    !git clone https://github.com/city96/ComfyUI-GGUF /content/ComfyUI/custom_nodes/ComfyUI-GGUF
if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI_GGUF'):
    !ln -sf /content/ComfyUI/custom_nodes/ComfyUI-GGUF /content/ComfyUI/custom_nodes/ComfyUI_GGUF 2>/dev/null || true

# 4. Hubungkan custom nodes bawaan mosdif ke ComfyUI
!mkdir -p /content/ComfyUI/custom_nodes
!cp -r /content/mosdif/comfy_nodes/* /content/ComfyUI/custom_nodes/ 2>/dev/null || true
print('Setup workspace dan ComfyUI selesai!')

### 3. Install Dependensi mosdif
Menginstall pustaka pemrosesan video, ONNX Runtime GPU, GGUF, dan percepatan attention.

In [ ]:
!pip install -q av opencv-python mmengine huggingface_hub onnxruntime-gpu sageattention gguf
!pip install -q --no-build-isolation nvidia-vfx --index-url https://pypi.nvidia.com 2>/dev/null || true
print('Semua dependensi mosdif berhasil terinstall!')

### 4. Unduh Model & Bobot (LTX-2.5 Uncensored GGUF, VAE, & VSR)
Mengunduh model deteksi mozaik RF-DETR (`rfdetr.onnx`), model BasicVSR++, DiT LTX-2.5 Uncensored GGUF (`ltx25_uncensored_v1.1-Q4_K_M.gguf`), Gemma-4 INT8 Text Encoder, dan Video VAE.

*Catatan: LoRA sudah di-bake langsung di dalam checkpoint model (Eros10 NSFW & DMD Distilled).* 

In [ ]:
import os
import threading
from pathlib import Path

!mkdir -p /content/mosdif/models
%cd /content/mosdif

# 1. Download Detector (rfdetr.onnx) jika belum ada
if not os.path.exists('/content/mosdif/models/rfdetr.onnx'):
    print('Mengunduh detector rfdetr.onnx...')
    !wget -q -O /content/mosdif/models/MosaicDiff-windows.7z "https://github.com/dotaku22/MosaicDiff/releases/download/v1.0.1/MosaicDiff-windows.7z"
    !7z e /content/mosdif/models/MosaicDiff-windows.7z models/rfdetr.onnx -o/content/mosdif/models/ -y > /dev/null
    !rm -f /content/mosdif/models/MosaicDiff-windows.7z
    print('Detector rfdetr.onnx siap!')

# 2. Konfigurasi path ComfyUI ke settings
from mosaicdiff.settings import Settings
from mosaicdiff.fetch import ensure_weights

settings = Settings.load()
settings.paths['comfy_root'] = '/content/ComfyUI'
settings.paths['comfy_python'] = '/usr/bin/python3'
settings.save()

# 3. Unduh model Hugging Face (BasicVSR, LTX-2.5 GGUF, Gemma-4, VAE) jika belum ada
print('\nMemeriksa dan mengunduh bobot model dari Hugging Face...')
ensure_weights(settings, print, threading.Event())
print('\nSemua model terverifikasi dan siap digunakan!')

### 5. Input Video & Pengaturan Parameter (`#@param`)
Pilih video yang ingin direstorasi dan atur parameternya pada panel formulir di samping kanan.
Cell ini juga langsung menampilkan **Preview Video Input** menggunakan Base64 encoding HTML5.

In [ ]:
#@title ⚙️ Formulir Input Video & Pengaturan Parameter { run: "auto" }

import os
import base64
from pathlib import Path
from IPython.display import HTML, display

# Helper untuk menampilkan video secara inline menggunakan Base64
def display_video_b64(video_path: str, title: str = 'Video Preview', width: int = 540):
    if not os.path.isfile(video_path):
        print(f'File tidak ditemukan: {video_path}')
        return
    with open(video_path, 'rb') as f:
        video_bytes = f.read()
    b64_str = base64.b64encode(video_bytes).decode('utf-8')
    ext = Path(video_path).suffix.lstrip('.').lower()
    mime = 'video/mp4' if ext in ['mp4', 'm4v'] else f'video/{ext}'
    
    html_code = f'''
    <div style="margin: 12px 0; padding: 12px; background-color: #1a1c23; border-radius: 8px; border: 1px solid #2e3440; display: inline-block;">
        <h4 style="color: #e0a15a; margin: 0 0 8px 0; font-family: sans-serif;">{title}</h4>
        <video width="{width}" controls style="border-radius: 6px; max-height: 420px; background: #000;">
            <source src="data:{mime};base64,{b64_str}" type="{mime}">
            Browser Anda tidak mendukung tag video HTML5.
        </video>
        <p style="color: #8892b0; margin: 6px 0 0 0; font-size: 12px; font-family: monospace;">{Path(video_path).name} ({len(video_bytes)/(1024*1024):.2f} MB)</p>
    </div>
    '''
    display(HTML(html_code))

# ==============================================================================
# PARAMETER INPUT & PENGATURAN RESTORASI
# ==============================================================================
video_input_path = "/content/input.mp4" #@param {type:"string"}
upload_video_langsung = False #@param {type:"boolean"}

#@markdown ### 🔍 Mode Refinement / Detailer (Berdasarkan Referensi BasicVSR++)
#@markdown `denoise_strength` mengatur porsi noise vs referensi BasicVSR++:
#@markdown - `0.30 - 0.40`: Sangat setia pada hasil BasicVSR++, hanya membersihkan blur & menambah ketajaman.
#@markdown - `0.40 - 0.45` (Rekomendasi Utama): Detail tekstur kulit/pori sangat realistis & natural tanpa mengubah bentuk anatomi asli.
#@markdown - `0.55+`: Rekonstruksi lebih agresif jika hasil BasicVSR++ sangat buram.
denoise_strength = 0.4 #@param {type:"slider", min:0.15, max:0.85, step:0.05}
sampling_steps = 6 #@param {type:"slider", min:2, max:12, step:1}
generation_resolution = 512 #@param [384, 512, 640, 768, 1024] {type:"raw"}
window_seconds = 3 #@param {type:"slider", min:1, max:8, step:1}
seed = 42 #@param {type:"integer"}

#@markdown ### 🎨 Prompt (Opsional - Default Otomatis Universal)
#@markdown Karena LTX-2.5 bekerja sebagai *refiner* atas referensi BasicVSR++, Anda TIDAK PERLU menulis prompt rumit.
#@markdown Prompt default ini akan otomatis mendetailkan tekstur kulit & kejernihan video secara natural:
prompt = "high quality, natural skin texture, sharp focus, 4k photorealistic, seamless natural detail, realistic lighting" #@param {type:"string"}
output_folder = "/content/output" #@param {type:"string"}

os.makedirs(output_folder, exist_ok=True)

# Fitur upload langsung lewat dialog Colab
if upload_video_langsung:
    try:
        from google.colab import files
        print('Pilih file video dari komputer Anda:')
        uploaded = files.upload()
        if uploaded:
            video_input_path = str(Path('/content') / list(uploaded.keys())[0])
            print(f'File berhasil diunggah: {video_input_path}')
    except Exception as err:
        print(f'Upload dialog: {err}')

print('\n' + '='*55)
print(f'Input Video     : {video_input_path}')
print(f'Mode Refiner    : Denoise={denoise_strength:.2f} (Referensi BasicVSR++ terjaga)')
print(f'Sampling Steps  : {sampling_steps} (Turbo)')
print(f'Resolusi Generasi: {generation_resolution}px')
print(f'Panjang Window  : {window_seconds}s')
print(f'Seed            : {seed}')
print(f'Prompt LTX-2.5  : {prompt}')
print(f'Folder Output   : {output_folder}')
print('='*55 + '\n')

if os.path.isfile(video_input_path):
    print('▶️ Menampilkan Preview Video Input:')
    display_video_b64(video_input_path, title='Preview Video Input (Sebelum Restorasi)')
else:
    print(f'⚠️ Video "{video_input_path}" belum ditemukan. Unggah video ke folder /content/ atau centang "upload_video_langsung".')


### 6. Jalankan Restorasi Video
Mengeksekusi pipeline deteksi mozaik, BasicVSR++, dan LTX-2.5 Uncensored Turbo inpainting.

In [ ]:
#@title ⚡ Eksekusi Restorasi Video { run: "auto" }

import sys
import time
import threading
import importlib
from pathlib import Path

# Bersihkan cache module mosaicdiff dari sys.modules agar impor baru selalu sinkron
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('mosaicdiff'):
        sys.modules.pop(mod_name, None)
from mosaicdiff.settings import Settings
from mosaicdiff.pipeline import process_video, unique_path

input_file = Path(video_input_path)
if not input_file.is_file():
    raise FileNotFoundError(f'File video input "{input_file}" tidak ditemukan! Silakan atur path pada Cell 5.')

out_dir = Path(output_folder)
out_dir.mkdir(parents=True, exist_ok=True)
dest_file = unique_path(out_dir / f'{input_file.stem}_restored.mp4')

# Update konfigurasi settings
settings = Settings.load()
settings.paths['comfy_root'] = '/content/ComfyUI'
settings.paths['comfy_python'] = '/usr/bin/python3'
settings.prompt = str(prompt).strip()
settings.seed = int(seed)
settings.denoise = float(denoise_strength)
settings.ltx_steps = int(sampling_steps)
settings.ltx_resolution = int(generation_resolution)
settings.ltx_seconds = int(window_seconds)
settings.output_dir = str(out_dir)
settings.save()

print('='*60)
print(f'Memulai Restorasi Video: {input_file.name}')
print(f'Tujuan File Output     : {dest_file.name}')
print(f'Parameter              : Denoise={settings.denoise:.2f}, Steps={settings.ltx_steps}, Res={settings.ltx_resolution}px, Seed={settings.seed}')
print(f'Mode                   : Latent Refiner (Menggunakan BasicVSR++ sebagai referensi)')
print(f'Prompt LTX-2.5         : "{settings.prompt}"')
print('='*60)

cancel_event = threading.Event()
start_time = time.time()

def log_cb(msg: str):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

def progress_cb(fraction: float, desc: str = ''):
    percent = fraction * 100
    bar_len = 25
    filled = int(round(bar_len * fraction))
    bar = '█' * filled + '-' * (bar_len - filled)
    sys.stdout.write(f'\r[{bar}] {percent:.1f}% - {desc}   ')
    sys.stdout.flush()

try:
    written_video = process_video(
        source=input_file,
        destination=dest_file,
        settings=settings,
        log=log_cb,
        progress=progress_cb,
        cancel=cancel_event,
    )
    print('\n')
    elapsed = time.time() - start_time
    print('='*60)
    print(f'✅ Restorasi Berhasil Selesai dalam {elapsed/60:.1f} menit ({elapsed:.0f} detik)!')
    print(f'File Tersimpan di : {written_video}')
    output_video_path = str(written_video)
    print('='*60)
except Exception as e:
    print(f'\n❌ Terjadi error saat pemrosesan: {e}')
    raise


### 7. Preview Video Hasil (Base64) & Unduh File
Menampilkan video hasil restorasi langsung di notebook dan menyediakan opsi download langsung ke komputer lokal.

In [ ]:
#@title 📺 Preview Video Hasil (Base64) & Unduh { run: "auto" }

import os
from pathlib import Path

unduh_ke_komputer = False #@param {type:"boolean"}

if 'output_video_path' in locals() and os.path.isfile(output_video_path):
    print('▶️ Menampilkan Preview Video Hasil:')
    display_video_b64(output_video_path, title='Preview Video Hasil (Sesudah Restorasi)', width=540)
    
    if unduh_ke_komputer:
        try:
            from google.colab import files
            print(f'Mengunduh {Path(output_video_path).name} ke komputer lokal...')
            files.download(output_video_path)
        except Exception as err:
            print(f'Gagal mendownload otomatis: {err}')
else:
    print('⚠️ Belum ada video hasil yang diproses. Silakan jalankan Cell 6 terlebih dahulu.')
